## How to run (English)

1. Download the dataset from Mendeley Data (doi:10.17632/3mbnb82mxd.2) and unzip it; you need the folder `Tomato Dataset/Dataset (raw)`.
2. **Google Colab (recommended, T4 GPU):** upload `Tomato Dataset` to your Google Drive (`MyDrive/Tomato Dataset`), then run
   `!git clone https://github.com/<user>/tomato-leaf /content/tomato-leaf` in a cell and open this notebook. Runtime → Change runtime type → T4 GPU.
3. **Locally:** set `TOMATO_DATA=/path/to/Tomato Dataset` (and optionally `TOMATO_REPO`, `TOMATO_OUT`) and start Jupyter from `notebooks/`.
4. Only cell **2) Paths** contains paths. New results go to `OUT_DIR/results` (never to the repository's `results/`). The notebook resumes:
   finished (model, seed, fold) runs found there are skipped.
5. Quick check without a GPU: set `TOMATO_SMOKE=1` (2 small models, 2 folds, 2 epochs, 64 px, no pretrained weights).
6. Full run: 20 models × 25 CV runs + 100 final runs took ≈ 16 h of training time on a T4 (13.8 h CV + 1.9 h final, from the `minutes` column of results/cv_runs.csv and final_test_runs.csv; caching and evaluation add more). Use `RUN_ONLY = ['PVTv2-B2']` in cell 1 to run selected models.

Comments inside the code cells are in Turkish; variable names and outputs are in English.


# Domates yaprağı hastalığı — 10 CNN + 5 ViT + 5 hibrit, sızıntısız 5-katlı grup CV × 5 seed (v6)

**Veri:** Pakistan domates yaprağı veri seti (Mendeley 10.17632/3mbnb82mxd, CC BY 4.0), yalnızca `Dataset (raw)` (830 ham görüntü).
Sızıntı denetimi (MD5 + pHash + DINOv2 + çekim oturumu) sonrası **748 görüntü / 682 bağımsız grup**; 36 kopya ve 47 etiket-çakışmalı görüntü çıkarıldı.

**Protokol**
1. Test kümesi (150 görüntü, grup-ayrık) model seçimine kadar **hiç kullanılmaz**.
2. Train/val (598): StratifiedGroupKFold 5 kat × seed 0–4 → model başına 25 eğitim.
3. En iyi model **yalnızca CV ortalama macro-F1** ile seçilir (istatistiksel eşitlikte daha az parametreli olan).
4. Her model 598 görüntünün tamamında 5 seed ile yeniden eğitilir (epoch = CV en iyi epoch medyanı) ve testte **bir kez** değerlendirilir.
5. Augmentasyon yalnızca train'de, eğitim sırasında anlık uygulanır.

**Pilot modu:** `RUN_ONLY` (hücre 1) tek bir model listesi içeriyorsa yalnızca o model çalışır; tam deney için `None` yapın. Pilotun sonuçları `results/` altında kalır ve tam çalışmada tekrar edilmez.

**v3:** v2'deki 10 CNN'ye 5 ViT (C grubu) eklendi.

**v4:** 5 hibrit CNN–dönüştürücü modeli (D grubu: CoAtNet-0, MaxViT-T, FastViT-SA12, MobileViTv2-1.0, MambaVision-T) eklendi. MambaVision-T timm'de olmadığından ağırlıkları Hugging Face `nvidia/MambaVision-T-1K` deposundan (sabit revizyon) yüklenir; `mamba_ssm` CUDA çekirdeği yerine aynı formülü uygulayan saf-PyTorch seçici tarama (`selective_scan_ref`) kullanılır (sonuç aynı, yalnızca daha yavaş). Model seçimi (hücre 8) artık 20 modelin tamamı üzerinden yapılır.

**v6:** Colab torch 2.11+cu130 getirdiği için (mamba-ssm'in hazır çekirdeği yok) hücre 0 torch'u 2.10.0'a (PyPI, +cu128) sabitler; böylece MambaVision resmî CUDA çekirdeğiyle çalışır. Hücre 0'dan önce torch yüklenmişse oturumu yeniden başlatın.

**v5:** MambaVision-T GPU'da resmî **mamba-ssm CUDA çekirdeği** (`selective_scan_cuda`, v2.3.2.post1 hazır wheel, `--no-deps`) ile çalışır. Çekirdek kullanılmadan önce saf-PyTorch referansla ileri+geri yönde karşılaştırılır (göreli fark < 1e-3); kurulamaz veya uyuşmazsa otomatik olarak referans taramaya dönülür ve bu yazdırılır. `RERUN = ['MambaVision-T']` (hücre 1) bu modelin eski (referans taramayla elde edilmiş) sonuçlarını `results/_yedek_*` altına taşıyıp CUDA çekirdeğiyle **bir kez** yeniden eğitir; çekirdek hazır değilse hiçbir sonuç silinmez. Verimlilik tablosuna `scan_backend` sütunu eklendi (FLOP sayımı her durumda referans formülle yapılır).

`results/` klasöründe bitmiş (model, seed, kat) ve (model, seed) eğitimleri otomatik atlanır; yalnızca eksikler (ve RERUN'daki modeller) eğitilir.

**Çalıştırmadan önce:** Drive'da *Benimle paylaşılanlar → Tomato Dataset* klasörüne sağ tık → *Kısayol ekle → Drive'ım*.
Çalışma zamanı: *Runtime → Change runtime type → GPU (T4)*. Bağlantı koparsa notebook'u yeniden çalıştırın; biten eğitimler atlanır.

In [ ]:
# 0) Kurulum
import sys, subprocess, importlib, importlib.util
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'timm>=1.0.9', 'einops', 'safetensors', 'huggingface_hub'], check=True)

# MambaVision'ın resmî CUDA çekirdeği (mamba-ssm 2.3.2.post1) yalnızca torch 2.6–2.10 için hazır derlenmiş olarak yayımlanıyor.
# Colab daha yeni bir torch (ör. 2.11+cu130) getiriyorsa, torch BAŞKA HİÇBİR ŞEY YÜKLENMEDEN önce PyPI'deki torch 2.10.0 (+cu128)
# ve torchvision 0.25.0'a sabitlenir (T4 sürücüsü CUDA 12.8 ile uyumludur). Bu oturumda 20 modelin verimliliği de aynı sürümle ölçülür.
MAMBA_TORCH_PIN = True
if IN_COLAB and MAMBA_TORCH_PIN:
    from importlib.metadata import version as _ver
    _tv = tuple(int(x) for x in _ver('torch').split('+')[0].split('.')[:2])
    if not ((2, 6) <= _tv <= (2, 10)):
        if 'torch' in sys.modules:
            raise RuntimeError('torch zaten yüklenmiş: Runtime → Restart session yapıp notebook\'u baştan çalıştırın.')
        print(f'torch {_ver("torch")} için mamba-ssm çekirdeği yok → torch 2.10.0 + torchvision 0.25.0 kuruluyor (birkaç dakika)...', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.10.0', 'torchvision==0.25.0'], check=True)
        importlib.invalidate_caches()
import torch as _t; print('torch', _t.__version__, '| CUDA', _t.version.cuda, '| GPU', _t.cuda.get_device_name() if _t.cuda.is_available() else '-')

In [ ]:
# 1) Ayarlar
import os, json, time, math, glob, hashlib, random, shutil
import numpy as np, pandas as pd

SMOKE = bool(int(os.environ.get('TOMATO_SMOKE', '0')))   # yerel kısa deneme modu

MODELS = {   # ad: (timm etiketi, grup) — hepsi yalnızca ImageNet-1k ön-eğitimli
    'EfficientNet-B0':         ('efficientnet_b0.ra_in1k',                 'A: klasik TL'),
    'ResNet50':                ('resnet50.a1_in1k',                        'A: klasik TL'),
    'InceptionV3':             ('inception_v3.tv_in1k',                    'A: klasik TL'),
    'DenseNet121':             ('densenet121.tv_in1k',                     'A: klasik TL'),
    'MobileNetV2':             ('mobilenetv2_100.ra_in1k',                 'A: klasik TL'),
    'ConvNeXtV2-Tiny':         ('convnextv2_tiny.fcmae_ft_in1k',           'B: verimli CNN'),
    'EfficientNetV2-S':        ('tf_efficientnetv2_s.in1k',                'B: verimli CNN'),
    'InceptionNeXt-Tiny':      ('inception_next_tiny.sail_in1k',           'B: verimli CNN'),
    'MobileNetV4-Conv-M':      ('mobilenetv4_conv_medium.e500_r224_in1k',  'B: verimli CNN'),
    'GhostNetV2-1.0':          ('ghostnetv2_100.in1k',                     'B: verimli CNN'),
    'ViT-S/16':                ('vit_small_patch16_224.augreg_in1k',       'C: ViT'),
    'DeiT III-S':              ('deit3_small_patch16_224.fb_in1k',         'C: ViT'),
    'Swin-T':                  ('swin_tiny_patch4_window7_224.ms_in1k',    'C: ViT'),
    'PVTv2-B2':                ('pvt_v2_b2.in1k',                          'C: ViT'),
    'XCiT-S12/16':             ('xcit_small_12_p16_224.fb_in1k',           'C: ViT'),
    'CoAtNet-0':               ('coatnet_0_rw_224.sw_in1k',                'D: hibrit'),
    'MaxViT-T':                ('maxvit_tiny_tf_224.in1k',                 'D: hibrit'),
    'FastViT-SA12':            ('fastvit_sa12.apple_in1k',                 'D: hibrit'),
    'MobileViTv2-1.0':         ('mobilevitv2_100.cvnets_in1k',             'D: hibrit'),
    'MambaVision-T':           ('hf:nvidia/MambaVision-T-1K',              'D: hibrit'),   # HF, NVIDIA Source Code License-NC
}
CLASSES = ['Early blight', 'Late blight', 'Septoria leaf spot', 'Leaf mold', 'Yellow leaf curl virus', 'Healthy']
SEEDS, FOLDS = [0, 1, 2, 3, 4], [0, 1, 2, 3, 4]
CFG = dict(img=224, cache_short=256, batch=32, max_epochs=40, patience=7, warmup=3,
           lr_backbone=1e-4, lr_head=1e-3, wd=0.05, label_smoothing=0.1, workers=2, amp=True)
PRETRAINED = True
if SMOKE:   # yalnızca kodun uçtan uca çalıştığını doğrulamak için
    MODELS = {k: MODELS[k] for k in ['MobileNetV2', 'GhostNetV2-1.0']}
    SEEDS, FOLDS = [0], [0, 1]
    CFG.update(img=64, cache_short=72, batch=16, max_epochs=2, patience=1, warmup=1, workers=0, amp=False)
    PRETRAINED = False
RUN_ONLY = None   # örn. ['EfficientNet-B0'] = yalnız bu model; None = tüm modeller
RERUN = []   # [] = yeniden eğitim yok; örn. ['MambaVision-T']: bu modelin eski CV/final sonuçları results/_yedek_* altına taşınır ve model yeniden eğitilir (bir kez)
if RUN_ONLY and not SMOKE:
    MODELS = {k: MODELS[k] for k in RUN_ONLY}
print(len(MODELS), 'model:', list(MODELS), '|', len(SEEDS), 'seed |', len(FOLDS), 'kat | SMOKE =', SMOKE)

In [ ]:
# 2) Paths — the ONLY cell you need to edit
#   REPO_DIR : this repository (contains splits/ and results/)
#   DATA_ROOT: the Mendeley folder "Tomato Dataset" (must contain "Dataset (raw)")
#   OUT_DIR  : where NEW runs are written. Kept separate from the paper's results/ on purpose:
#              the notebook resumes by skipping runs it finds in OUT_DIR/results.
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR  = os.environ.get('TOMATO_REPO', '/content/tomato-leaf')     # !git clone https://github.com/<user>/tomato-leaf /content/tomato-leaf
    DATA_ROOT = os.environ.get('TOMATO_DATA', '/content/drive/MyDrive/Tomato Dataset')
    OUT_DIR   = os.environ.get('TOMATO_OUT',  '/content/drive/MyDrive/tomato-leaf-runs')
    CACHE = '/content/cache_raw'
else:
    REPO_DIR  = os.environ.get('TOMATO_REPO', os.path.abspath('..'))
    DATA_ROOT = os.environ['TOMATO_DATA']
    OUT_DIR   = os.environ.get('TOMATO_OUT', os.path.join(REPO_DIR, 'runs'))
    CACHE = os.path.join(OUT_DIR, '_cache')
assert os.path.isdir(os.path.join(DATA_ROOT, 'Dataset (raw)')), f"'Dataset (raw)' not found under DATA_ROOT = {DATA_ROOT}"
SPLIT_CSV = os.path.join(REPO_DIR, 'splits', 'raw_clean_group_split.csv')
assert os.path.exists(SPLIT_CSV), f'split file not found: {SPLIT_CSV}'
PROJ = OUT_DIR
RES = os.path.join(OUT_DIR, 'results'); os.makedirs(RES, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT); print('SPLIT_CSV =', SPLIT_CSV); print('RESULTS   =', RES)


In [ ]:
# 3) Bölme dosyası + ham görüntülerin doğrulanması (MD5) + yerel önbellek (kısa kenar 256 px)
from PIL import Image
df = pd.read_csv(SPLIT_CSV)
assert len(df) == 748 and df.split.value_counts().to_dict() == {'trainval': 598, 'test': 150}
assert df.groupby('group').split.nunique().max() == 1, 'bir grup hem testte hem trainval’de!'
assert df[df.split == 'trainval'].groupby('group').fold.nunique().max() == 1, 'bir grup birden fazla katta!'

def _long(p): return ('\\\\?\\' + os.path.abspath(p)) if os.name == 'nt' else p
def cache_one(row):
    src = _long(os.path.join(DATA_ROOT, row.relpath))
    dst = os.path.join(CACHE, f'{row.Index:04d}.jpg')
    raw = open(src, 'rb').read()
    ok = hashlib.md5(raw).hexdigest() == row.md5
    if not os.path.exists(dst):
        im = Image.open(src).convert('RGB'); s = CFG['cache_short'] / min(im.size)
        im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BICUBIC).save(dst, quality=95)
    return dst, ok

from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(16) as ex: res = list(ex.map(cache_one, df.itertuples()))
df['path'] = [r[0] for r in res]; md5_ok = np.array([r[1] for r in res])
print(f'önbellek: {len(df)} görüntü, {time.time()-t0:.0f} s | MD5 eşleşmeyen: {(~md5_ok).sum()}')
assert md5_ok.all(), 'Drive’daki dosyalar denetlenen kopyayla aynı değil — durun ve kontrol edin.'

In [ ]:
# 4) Sızıntı kontrolü: test ile trainval arasında birebir (MD5) veya döndürme/çevirmeye dayanıklı pHash (d<=10) eşleşmesi olmamalı
from scipy.fft import dctn
def dihedral_phash(path):
    g = np.asarray(Image.open(path).convert('L').resize((32, 32), Image.BILINEAR), dtype=np.float32)
    out = []
    for k in range(4):
        for fl in (False, True):
            a = np.rot90(g, k); a = a[:, ::-1] if fl else a
            d = dctn(a, norm='ortho')[:8, :8].ravel(); out.append(np.packbits(d > np.median(d[1:])).view('>u8')[0])
    return out
H = np.array([dihedral_phash(p) for p in df.path], dtype=np.uint64)
LUT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
te, tv = np.where(df.split == 'test')[0], np.where(df.split == 'trainval')[0]
dmin = LUT[(H[te, 0][:, None, None] ^ H[tv][None]).view(np.uint8).reshape(len(te), len(tv), 8, 8)].sum(-1).min(-1)
n_md5 = len(set(df.md5.iloc[te]) & set(df.md5.iloc[tv]))
print(f'test↔trainval: ortak MD5 = {n_md5} | en küçük pHash mesafesi = {dmin.min()} (eşik 10)')
assert n_md5 == 0 and dmin.min() > 10, 'SIZINTI: eğitim başlatılmadı.'

In [ ]:
# 5) Veri yükleyiciler — augmentasyon YALNIZCA train'de
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')
IMGS = {p: Image.open(p).convert('RGB') for p in df.path}          # 748 küçük görüntü RAM'de

def tfms(mean, std, train):
    s = CFG['img']
    if train:
        return T.Compose([T.RandomResizedCrop(s, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                          T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(mean, std)])
    return T.Compose([T.Resize(s), T.CenterCrop(s), T.ToTensor(), T.Normalize(mean, std)])

class DS(Dataset):
    def __init__(self, sub, tf): self.p, self.y, self.tf = sub.path.tolist(), sub.label.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(IMGS[self.p[i]]), self.y[i]

def loaders(train_df, eval_df, mean, std, seed):
    g = torch.Generator().manual_seed(seed)
    tl = DataLoader(DS(train_df, tfms(mean, std, True)), CFG['batch'], shuffle=True, drop_last=len(train_df) > CFG['batch'],
                    num_workers=CFG['workers'], generator=g, pin_memory=DEV == 'cuda', persistent_workers=CFG['workers'] > 0)
    el = None if eval_df is None else DataLoader(DS(eval_df, tfms(mean, std, False)), 64, num_workers=CFG['workers'])
    return tl, el

In [ ]:
# 6) Model, eğitim ve değerlendirme fonksiyonları
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score

# --- MambaVision (HF ağırlıkları, transformers'sız yükleme). Seçici tarama: GPU'da mamba-ssm CUDA çekirdeği, yoksa saf-PyTorch referans ---
import sys, types, subprocess, importlib, torch, torch.nn as nn, torch.nn.functional as F
def selective_scan_ref(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False, return_last_state=False):
    with torch.autocast(device_type=u.device.type, enabled=False):   # tarama fp32'de
        dtype_in = u.dtype; u = u.float(); delta = delta.float()
        if delta_bias is not None: delta = delta + delta_bias[..., None].float()
        if delta_softplus: delta = F.softplus(delta)
        A, B, C = A.float(), B.float(), C.float()                       # A:(d,n)  B,C:(b,n,l)
        dA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A)); dBu = torch.einsum('bdl,bnl,bdl->bdln', delta, B, u)
        x = u.new_zeros(u.shape[0], A.shape[0], A.shape[1]); ys = []
        for i in range(u.shape[2]):
            x = dA[:, :, i] * x + dBu[:, :, i]; ys.append(torch.einsum('bdn,bn->bd', x, C[:, :, i]))
        y = torch.stack(ys, 2)
        out = y if D is None else y + u * D.float()[:, None]
        if z is not None: out = out * F.silu(z.float())
        out = out.to(dtype_in)
        return (out, x) if return_last_state else out
# --- mamba-ssm CUDA çekirdeği: Colab'daki torch/CUDA/Python sürümüne uyan hazır wheel --no-deps kurulur; yalnızca derlenmiş
#     `selective_scan_cuda` kullanılır (mamba_ssm'in Python paketi ve tilelang/triton bağımlılıkları yüklenmez → torch bozulmaz).
#     Autograd sarmalayıcısı mamba_ssm v2.3.2.post1 SelectiveScanFn'den alınmıştır (Tri Dao, Albert Gu; Apache-2.0).
MAMBA_SSM_VER = '2.3.2.post1'
SCAN_BACKEND = None          # 'cuda' (mamba_ssm çekirdeği) | 'ref' (saf PyTorch); ilk MambaVision kurulumunda belirlenir
_scan_cuda = None
class _SelectiveScanCuda(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False):
        u, delta, B, C = u.contiguous(), delta.contiguous(), B.contiguous(), C.contiguous()
        D = D.contiguous() if D is not None else None; z = z.contiguous() if z is not None else None
        ctx.squeeze_B, ctx.squeeze_C = B.dim() == 3, C.dim() == 3
        if B.dim() == 3: B = B.unsqueeze(1)                     # b n l -> b 1 n l
        if C.dim() == 3: C = C.unsqueeze(1)
        out, x, *rest = _scan_cuda.fwd(u, delta, A, B, C, D, z, delta_bias, delta_softplus)
        ctx.delta_softplus, ctx.has_z = delta_softplus, z is not None
        if not ctx.has_z: ctx.save_for_backward(u, delta, A, B, C, D, delta_bias, x); return out
        ctx.save_for_backward(u, delta, A, B, C, D, z, delta_bias, x, out); return rest[0]
    @staticmethod
    def backward(ctx, dout):
        if not ctx.has_z: (u, delta, A, B, C, D, delta_bias, x), z, out = ctx.saved_tensors, None, None
        else: u, delta, A, B, C, D, z, delta_bias, x, out = ctx.saved_tensors
        du, ddelta, dA, dB, dC, dD, ddelta_bias, *rest = _scan_cuda.bwd(
            u, delta, A, B, C, D, z, delta_bias, dout.contiguous(), x, out, None, ctx.delta_softplus, False)
        if ctx.squeeze_B: dB = dB.squeeze(1)
        if ctx.squeeze_C: dC = dC.squeeze(1)
        return (du, ddelta, dA, dB, dC, dD if D is not None else None, rest[0] if ctx.has_z else None,
                ddelta_bias if delta_bias is not None else None, None)
def selective_scan_fn(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False, return_last_state=False):
    # MambaVision bu fonksiyonu çağırır; arka uç SCAN_BACKEND ile seçilir.
    if SCAN_BACKEND == 'cuda' and u.is_cuda and not return_last_state:
        return _SelectiveScanCuda.apply(u, delta, A, B, C, D, z, delta_bias, delta_softplus)
    return selective_scan_ref(u, delta, A, B, C, D, z, delta_bias, delta_softplus, return_last_state)
def _mamba_wheel_url():
    tv = '.'.join(torch.__version__.split('+')[0].split('.')[:2]); cu = (torch.version.cuda or '').split('.')[0]
    abi = 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE'; py = f'cp{sys.version_info.major}{sys.version_info.minor}'
    return (f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_SSM_VER}/'
            f'mamba_ssm-{MAMBA_SSM_VER}%2Bcu{cu}torch{tv}cxx11abi{abi}-{py}-{py}-linux_x86_64.whl')
def _check_scan_cuda(seed=0):
    # CUDA çekirdeği ile saf-PyTorch referansı ileri + geri yönde karşılaştırılır (fp32).
    g = torch.Generator(device='cuda').manual_seed(seed); b, d, n, l = 2, 64, 8, 196
    r = lambda *s: torch.randn(*s, device='cuda', generator=g)
    u, dt, B, C = r(b, d, l), r(b, d, l), r(b, n, l), r(b, n, l)
    A, D, db = -torch.exp(r(d, n) * 0.5), r(d), r(d) * 0.1
    outs = []
    for fn in (lambda *a: _SelectiveScanCuda.apply(*a), lambda *a: selective_scan_ref(*a)):
        xs = [t.clone().requires_grad_(True) for t in (u, dt, A, B, C, D, db)]
        y = fn(xs[0], xs[1], xs[2], xs[3], xs[4], xs[5], None, xs[6], True); y.backward(torch.ones_like(y))
        outs.append([y.detach()] + [t.grad for t in xs])
    err = max(((a - c).abs().max() / (c.abs().max() + 1e-6)).item() for a, c in zip(*outs))
    return err
def _setup_scan_backend():
    global SCAN_BACKEND, _scan_cuda
    if SCAN_BACKEND is not None: return SCAN_BACKEND
    SCAN_BACKEND = 'ref'
    if not torch.cuda.is_available():
        print('MambaVision: GPU yok → saf-PyTorch tarama'); return SCAN_BACKEND
    try:
        import selective_scan_cuda as _m
    except ImportError:
        url = _mamba_wheel_url(); print('MambaVision: mamba-ssm CUDA çekirdeği kuruluyor →', url.split('/')[-1].replace('%2B', '+'))
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', url], capture_output=True, text=True)
        if r.returncode != 0:
            print('  ✗ kurulamadı (bu torch/CUDA/Python için hazır wheel yok olabilir) → saf-PyTorch tarama\n ', r.stderr.strip()[-400:]); return SCAN_BACKEND
        importlib.invalidate_caches()
        try: import selective_scan_cuda as _m
        except Exception as e: print('  ✗ çekirdek yüklenemedi →', repr(e)[:300], '→ saf-PyTorch tarama'); return SCAN_BACKEND
    _scan_cuda = _m
    try: err = _check_scan_cuda()
    except Exception as e: print('  ✗ çekirdek çalışmadı →', repr(e)[:300], '→ saf-PyTorch tarama'); return SCAN_BACKEND
    if err < 1e-3: SCAN_BACKEND = 'cuda'
    print(f"MambaVision tarama arka ucu: {SCAN_BACKEND}  (CUDA–referans en büyük göreli fark, ileri+geri: {err:.2e}; GPU: {torch.cuda.get_device_name()})")
    return SCAN_BACKEND
def _install_scan_stub():
    if 'mamba_ssm.ops.selective_scan_interface' in sys.modules: return
    names = ['mamba_ssm', 'mamba_ssm.ops', 'mamba_ssm.ops.selective_scan_interface']
    for n in names: sys.modules[n] = types.ModuleType(n)
    sys.modules[names[0]].ops = sys.modules[names[1]]; sys.modules[names[1]].selective_scan_interface = sys.modules[names[2]]
    sys.modules[names[2]].selective_scan_fn = selective_scan_fn; sys.modules[names[2]].selective_scan_ref = selective_scan_ref

class HFBackbone(nn.Module):
    def __init__(self, net, mean, std):
        super().__init__(); self.net = net; self.pretrained_cfg = dict(mean=tuple(mean), std=tuple(std))
    def forward(self, x): return self.net(x)
    def get_classifier(self): return self.net.head
_MV_REV = 'b1de77e17599566d98efb701c0231b1095dc3a67'   # HF deposunun sabitlenmiş sürümü
def _mambavision_class(repo):
    from huggingface_hub import hf_hub_download
    src = open(hf_hub_download(repo, 'modeling_mambavision.py', revision=_MV_REV), encoding='utf-8').read()
    src = src.split('class MambaVisionModel(PreTrainedModel)')[0]          # transformers sarmalayıcıları atılır
    src = '\n'.join(l for l in src.splitlines() if 'from transformers' not in l and 'configuration_mambavision' not in l)
    _install_scan_stub(); ns = {'__name__': 'mambavision_src'}; exec(compile(src, 'modeling_mambavision.py', 'exec'), ns)
    return ns['MambaVision']
def make_model(tag, pretrained, num_classes):
    if not tag.startswith('hf:'):
        return timm.create_model(tag, pretrained=pretrained, num_classes=num_classes)
    import json as _json
    from huggingface_hub import hf_hub_download
    _setup_scan_backend()
    repo = tag[3:]; cfg = _json.load(open(hf_hub_download(repo, 'config.json', revision=_MV_REV)))
    net = _mambavision_class(repo)(depths=cfg['depths'], num_heads=cfg['num_heads'], window_size=cfg['window_size'], dim=cfg['dim'],
                                   in_dim=cfg['in_dim'], mlp_ratio=cfg['mlp_ratio'], num_classes=1000)
    if pretrained:
        from safetensors.torch import load_file
        sd = load_file(hf_hub_download(repo, 'model.safetensors', revision=_MV_REV))
        sd = {k[len('model.'):]: v for k, v in sd.items() if k.startswith('model.')}
        missing, unexpected = net.load_state_dict(sd, strict=False)
        assert not unexpected and not [k for k in missing if not k.startswith('head')], (missing, unexpected)
    net.head = nn.Linear(net.head.in_features, num_classes)
    nn.init.trunc_normal_(net.head.weight, std=0.02); nn.init.zeros_(net.head.bias)
    return HFBackbone(net, cfg['mean'], cfg['std'])

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True      # hız için; sonuçlar bit düzeyinde birebir tekrarlanmaz

def build(tag):
    m = make_model(tag, PRETRAINED, len(CLASSES))
    cfg = m.pretrained_cfg; return m.to(DEV), cfg.get('mean', (0.485, 0.456, 0.406)), cfg.get('std', (0.229, 0.224, 0.225))

def metrics(y, p):
    return dict(macro_f1=f1_score(y, p, average='macro'), acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                mcc=matthews_corrcoef(y, p), kappa=cohen_kappa_score(y, p))

@torch.no_grad()
def predict(m, dl):
    m.eval(); P = []
    for x, _ in dl:
        with torch.autocast(DEV, enabled=CFG['amp'] and DEV == 'cuda'): P.append(m(x.to(DEV, non_blocking=True)).float().softmax(1).cpu())
    return torch.cat(P).numpy()

def train_one(tag, train_df, val_df, seed, epochs=None):
    # val_df verilirse erken durdurma val macro-F1 ile; verilmezse sabit `epochs` kadar eğitilir (final model).
    set_seed(seed); m, mean, std = build(tag); tl, vl = loaders(train_df, val_df, mean, std, seed)
    head = set(id(p) for p in m.get_classifier().parameters())
    opt = torch.optim.AdamW([{'params': [p for p in m.parameters() if id(p) not in head], 'lr': CFG['lr_backbone']},
                             {'params': [p for p in m.parameters() if id(p) in head], 'lr': CFG['lr_head']}], weight_decay=CFG['wd'])
    E = epochs or CFG['max_epochs']; steps = E * len(tl); wsteps = CFG['warmup'] * len(tl)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / wsteps if s < wsteps else 0.5 * (1 + math.cos(math.pi * (s - wsteps) / max(1, steps - wsteps))))
    cnt = np.bincount(train_df.label, minlength=len(CLASSES)); w = torch.tensor(cnt.sum() / (len(CLASSES) * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG['label_smoothing'])
    scaler = torch.amp.GradScaler(enabled=CFG['amp'] and DEV == 'cuda')
    best, best_ep, best_state, bad, hist = -1, 0, None, 0, []
    for ep in range(1, E + 1):
        m.train(); tot = 0
        for x, y in tl:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast(DEV, enabled=CFG['amp'] and DEV == 'cuda'): loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tot += loss.item() * len(y)
        if vl is None: continue
        f1 = f1_score(val_df.label, predict(m, vl).argmax(1), average='macro'); hist.append(round(f1, 4))
        if f1 > best: best, best_ep, bad = f1, ep, 0; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG['patience']: break
    if best_state is not None: m.load_state_dict(best_state)
    return m, mean, std, best_ep, hist

In [ ]:
# 7) 5-katlı grup CV × 5 seed (kaldığı yerden devam eder). Test kümesine dokunulmaz.
CV_CSV = os.path.join(RES, 'cv_runs.csv')
TV = df[df.split == 'trainval'].reset_index(drop=True)

# İsteğe bağlı yeniden eğitim: RERUN'daki her model için eski sonuçlar BİR KEZ yedeklenip çıkarılır (işaret dosyası: rerun_<model>.json)
for _name in [n for n in RERUN if n in MODELS]:
    _safe = _name.replace('/', '-'); _mark = os.path.join(RES, f'rerun_{_safe}.json')
    if os.path.exists(_mark): print(f'RERUN {_name}: daha önce sıfırlandı, kaldığı yerden devam ediliyor'); continue
    if MODELS[_name][0].startswith('hf:nvidia/MambaVision') and _setup_scan_backend() != 'cuda':
        print(f'⚠ RERUN {_name}: CUDA çekirdeği hazır değil → yeniden eğitim YAPILMADI, eski sonuçlar korundu'); continue
    _bk = os.path.join(RES, '_yedek_' + time.strftime('%Y%m%d_%H%M%S') + '_' + _safe); os.makedirs(_bk, exist_ok=True)
    for _f in ['cv_runs.csv', 'final_test_runs.csv']:
        _p = os.path.join(RES, _f)
        if os.path.exists(_p):
            shutil.copy(_p, os.path.join(_bk, _f)); _d = pd.read_csv(_p); _d[_d.model != _name].to_csv(_p, index=False)
    for _f in glob.glob(os.path.join(RES, 'test_probs', f'{_safe}__seed*.npy')): shutil.move(_f, os.path.join(_bk, os.path.basename(_f)))
    json.dump(dict(model=_name, backup=_bk, scan_backend=SCAN_BACKEND, time=time.strftime('%Y-%m-%d %H:%M:%S')), open(_mark, 'w'))
    print(f'RERUN {_name}: eski sonuçlar → {_bk}; model yeniden eğitilecek')

def model_report(name):
    # Bir modelin 5-kat validation özetini yazdırır: her seed için 5 katın ortalaması + genel ortalama ± std
    d = pd.read_csv(CV_CSV); d = d[d.model == name]
    per_seed = d.groupby('seed').agg(kat=('fold', 'nunique'), macro_f1=('macro_f1', 'mean'), acc=('acc', 'mean'), bal_acc=('bal_acc', 'mean'))
    print('=' * 78)
    print(f"✔ {name} TAMAMLANDI  ({len(d)} eğitim)  |  5-kat VAL macro-F1 = {d.macro_f1.mean():.4f} ± {d.macro_f1.std():.4f}"
          f"  |  acc = {d.acc.mean():.4f}  |  bal_acc = {d.bal_acc.mean():.4f}  |  medyan en iyi epoch = {d.best_epoch.median():.0f}")
    print('  seed bazında 5-kat ortalamaları:', '  '.join(f"s{s}: {r.macro_f1:.4f}" for s, r in per_seed.iterrows()))
    print('=' * 78, flush=True)

def n_done(name):
    if not os.path.exists(CV_CSV): return set()
    d = pd.read_csv(CV_CSV); d = d[d.model == name]; return set(zip(d.seed, d.fold))

finished, pending = [n for n in MODELS if len(n_done(n)) == len(SEEDS) * len(FOLDS)], [n for n in MODELS if len(n_done(n)) < len(SEEDS) * len(FOLDS)]
print(f'CV durumu: {len(finished)} model bitmiş, {len(pending)} model kaldı → kalan: {pending}')
for name in finished: model_report(name)          # önceki oturumlarda bitmiş modeller

for mi, name in enumerate(pending, 1):
    done = n_done(name); todo = [(s, f) for s in SEEDS for f in FOLDS if (s, f) not in done]
    print(f"\n▶ [{len(finished) + mi}/{len(MODELS)}] {name} başlıyor — {len(todo)} eğitim kaldı", flush=True)
    for seed, fold in todo:
        t0 = time.time(); tr, va = TV[TV.fold != fold], TV[TV.fold == fold]
        m, mean, std, best_ep, hist = train_one(MODELS[name][0], tr, va, seed)
        _, vl = loaders(tr, va, mean, std, seed); pv = predict(m, vl).argmax(1)
        row = dict(model=name, group=MODELS[name][1], seed=seed, fold=fold, best_epoch=best_ep, n_train=len(tr), n_val=len(va),
                   minutes=round((time.time() - t0) / 60, 2), val_hist=json.dumps(hist), **metrics(va.label.values, pv))
        pd.DataFrame([row]).to_csv(CV_CSV, mode='a', header=not os.path.exists(CV_CSV), index=False)
        print(f"   {name:20s} seed{seed} fold{fold}  val macroF1={row['macro_f1']:.4f}  ep*={best_ep}  {row['minutes']} dk", flush=True)
        if fold == FOLDS[-1] or len(n_done(name) & {(seed, f) for f in FOLDS}) == len(FOLDS):
            d = pd.read_csv(CV_CSV); d = d[(d.model == name) & (d.seed == seed)]
            if d.fold.nunique() == len(FOLDS): print(f"   → seed{seed} bitti: 5-kat val macro-F1 ort. = {d.macro_f1.mean():.4f}", flush=True)
        del m; torch.cuda.empty_cache() if DEV == 'cuda' else None
    model_report(name)
print('\nTÜM MODELLERİN CV EĞİTİMİ TAMAMLANDI.')

In [ ]:
# 8) CV özeti ve EN İYİ MODELİN SEÇİMİ (yalnızca CV verisiyle)
from scipy import stats
cv = pd.read_csv(CV_CSV); cv = cv[cv.model.isin(MODELS)]
def nparams(tag): return sum(p.numel() for p in make_model(tag, False, len(CLASSES)).parameters())
PARAMS = {n: nparams(t) for n, (t, _) in MODELS.items()}
summ = (cv.groupby(['model', 'group']).agg(n_runs=('macro_f1', 'size'), macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', 'std'),
        acc_mean=('acc', 'mean'), bal_acc_mean=('bal_acc', 'mean'), mcc_mean=('mcc', 'mean'), median_best_epoch=('best_epoch', 'median'))
        .reset_index().assign(params_M=lambda d: d.model.map(PARAMS) / 1e6).sort_values('macro_f1_mean', ascending=False))
try: display(summ.round(4))
except NameError: print(summ.round(4).to_string())

piv = cv.pivot_table(index=['seed', 'fold'], columns='model', values='macro_f1').dropna()
k, N = piv.shape[1], piv.shape[0]
if k >= 3 and N >= 2:
    fr = stats.friedmanchisquare(*[piv[c] for c in piv.columns])
    ranks = piv.rank(axis=1, ascending=False).mean().sort_values()
    q05 = stats.studentized_range.ppf(0.95, k, np.inf) / math.sqrt(2)   # Nemenyi kritik değeri (k=10 → 3.164)
    CD = q05 * math.sqrt(k * (k + 1) / (6 * N))
    print(f'Friedman χ²={fr.statistic:.2f}, p={fr.pvalue:.2e} | Nemenyi CD(α=0.05)={CD:.2f} | ortalama sıralar:'); print(ranks.round(2).to_string())
top = summ.model.iloc[0]
tie = [top]
for other in summ.model.iloc[1:]:
    if N >= 6:
        p = stats.wilcoxon(piv[top], piv[other]).pvalue
        if p >= 0.05: tie.append(other)
SELECTED = min(tie, key=lambda n: PARAMS[n])
sel = dict(top_by_mean=top, statistically_tied_with_top=tie, selected=SELECTED,
           rule='CV ortalama macro-F1 en yüksek; Wilcoxon p>=0.05 ile ayırt edilemeyenler arasından en az parametreli')
json.dump(sel, open(os.path.join(RES, 'selection.json'), 'w'), ensure_ascii=False, indent=1)
summ.to_csv(os.path.join(RES, 'cv_summary.csv'), index=False); print('\nSEÇİLEN MODEL:', sel)

In [ ]:
# 9) Final eğitim: her model 598 görüntünün tamamında 5 seed (epoch = CV en iyi epoch medyanı), testte TEK değerlendirme
FIN_CSV = os.path.join(RES, 'final_test_runs.csv'); PRED_DIR = os.path.join(RES, 'test_probs'); os.makedirs(PRED_DIR, exist_ok=True)
fin = pd.read_csv(FIN_CSV) if os.path.exists(FIN_CSV) else pd.DataFrame()
done = set(zip(fin.model, fin.seed)) if len(fin) else set()
TE = df[df.split == 'test'].reset_index(drop=True); TVall = df[df.split == 'trainval'].reset_index(drop=True)
EPOCHS = summ.set_index('model').median_best_epoch.round().astype(int).clip(lower=1).to_dict()
todo = [(n, s) for n in MODELS for s in SEEDS if (n, s) not in done]
print(f'final: {len(done)} bitti, {len(todo)} kaldı | epoch sayıları: {EPOCHS}')
for i, (name, seed) in enumerate(todo, 1):
    t0 = time.time(); m, mean, std, _, _ = train_one(MODELS[name][0], TVall, None, seed, epochs=EPOCHS[name])
    _, tl = loaders(TVall, TE, mean, std, seed); P = predict(m, tl); np.save(os.path.join(PRED_DIR, f"{name.replace('/','-')}__seed{seed}.npy"), P)
    row = dict(model=name, group=MODELS[name][1], seed=seed, epochs=EPOCHS[name], minutes=round((time.time() - t0) / 60, 2), **metrics(TE.label.values, P.argmax(1)))
    pd.DataFrame([row]).to_csv(FIN_CSV, mode='a', header=not os.path.exists(FIN_CSV), index=False)
    print(f"[{i}/{len(todo)}] {name:20s} seed{seed}  TEST macroF1={row['macro_f1']:.4f}")
    del m; torch.cuda.empty_cache() if DEV == 'cuda' else None

In [ ]:
# 10) Test sonuçları: tüm modeller + seçilen model (seed ortalaması ± std, seed-ensemble, bootstrap %95 GA, McNemar)
fin = pd.read_csv(FIN_CSV); fin = fin[fin.model.isin(MODELS)]; y = TE.label.values; rng = np.random.default_rng(0)
B = rng.integers(0, len(y), (2000, len(y))); ENS = {}
rows = []
for name in MODELS:
    fs = sorted(glob.glob(os.path.join(PRED_DIR, f"{name.replace('/','-')}__seed*.npy")))
    if not fs: continue
    pe = np.mean([np.load(f) for f in fs], 0).argmax(1); ENS[name] = pe
    boot = [f1_score(y[b], pe[b], average='macro') for b in B]
    f = fin[fin.model == name]
    rows.append(dict(model=name, group=MODELS[name][1], n_seeds=len(f), test_macro_f1_mean=f.macro_f1.mean(), test_macro_f1_std=f.macro_f1.std(),
                     test_acc_mean=f.acc.mean(), ens_macro_f1=f1_score(y, pe, average='macro'), ens_acc=accuracy_score(y, pe),
                     ens_f1_ci95=f'{np.percentile(boot, 2.5):.3f}–{np.percentile(boot, 97.5):.3f}', ens_mcc=matthews_corrcoef(y, pe)))
test = pd.DataFrame(rows).merge(summ[['model', 'macro_f1_mean', 'params_M']].rename(columns={'macro_f1_mean': 'cv_macro_f1'}), on='model')
test['selected_by_cv'] = test.model.eq(SELECTED)
from statsmodels.stats.contingency_tables import mcnemar
def mc(a, b):
    t = [[np.sum((a == y) & (b == y)), np.sum((a == y) & (b != y))], [np.sum((a != y) & (b == y)), np.sum((a != y) & (b != y))]]
    return mcnemar(t, exact=True).pvalue
test['mcnemar_p_vs_selected'] = [np.nan if n == SELECTED else mc(ENS[SELECTED], ENS[n]) for n in test.model]
test = test.sort_values('cv_macro_f1', ascending=False); test.to_csv(os.path.join(RES, 'test_summary.csv'), index=False)
print(test.round(4).to_string(index=False))
print(f'\nCV ile seçilen model: {SELECTED} → test macro-F1 (5 seed) = {test.set_index("model").loc[SELECTED, "test_macro_f1_mean"]:.4f} '
      f'± {test.set_index("model").loc[SELECTED, "test_macro_f1_std"]:.4f}; ensemble = {test.set_index("model").loc[SELECTED, "ens_macro_f1"]:.4f}')

In [ ]:
# 11) Verimlilik: parametre, GFLOPs (224×224), GPU gecikmesi (batch=1)
from torch.utils.flop_counter import FlopCounterMode
eff = []
for name, (tag, grp) in MODELS.items():
    m = make_model(tag, False, len(CLASSES)).to(DEV).eval(); x = torch.randn(1, 3, CFG['img'], CFG['img'], device=DEV)
    with torch.no_grad():
        fc = FlopCounterMode(display=False); _bk_scan = SCAN_BACKEND
        SCAN_BACKEND = 'ref' if _bk_scan else None    # FLOP sayımı einsum tabanlı referansla (CUDA çekirdeğini FlopCounter göremez)
        with fc: m(x)
        SCAN_BACKEND = _bk_scan
        for _ in range(10): m(x)
        if DEV == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); n = 50 if not SMOKE else 3
        for _ in range(n): m(x)
        if DEV == 'cuda': torch.cuda.synchronize()
    eff.append(dict(model=name, params_M=PARAMS[name] / 1e6, GFLOPs=fc.get_total_flops() / 2e9, latency_ms=(time.perf_counter() - t0) / n * 1e3, device=DEV, scan_backend=(SCAN_BACKEND if tag.startswith('hf:') else '')))
    del m
eff = pd.DataFrame(eff); eff.to_csv(os.path.join(RES, 'efficiency.csv'), index=False); print(eff.round(3).to_string(index=False))

In [ ]:
# 12) Figürler: CV dağılımı, CV–verimlilik, seçilen modelin test karışıklık matrisi
import matplotlib; matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
order = summ.model.tolist(); col = {'A: klasik TL': '#8C8C8C', 'B: verimli CNN': '#4C72B0', 'C: ViT': '#DD8452', 'D: hibrit': '#55A868'}
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.boxplot([cv[cv.model == n].macro_f1 for n in order], patch_artist=True, boxprops=dict(facecolor='white'), medianprops=dict(color='black'))
ax.set_xticks(range(1, len(order) + 1), order)
for i, n in enumerate(order, 1):
    v = cv[cv.model == n].macro_f1; ax.scatter(np.random.default_rng(i).normal(i, 0.06, len(v)), v, s=8, color=col[MODELS[n][1]], alpha=0.7, zorder=3)
ax.set_ylabel('CV macro-F1 (5 kat × 5 seed)'); ax.set_title(f'CV macro-F1 (seçilen: {SELECTED})', loc='left'); plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
fig.tight_layout(); fig.savefig(os.path.join(RES, 'fig_cv_boxplot.png'), dpi=300)

E = summ.merge(eff[['model', 'GFLOPs', 'latency_ms']], on='model')
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for _, r in E.iterrows():
    ax.scatter(r.GFLOPs, r.macro_f1_mean, s=20 + 4 * r.params_M, color=col[r.group], edgecolor='black' if r.model == SELECTED else 'none', zorder=3)
    ax.annotate(r.model, (r.GFLOPs, r.macro_f1_mean), fontsize=7, xytext=(4, 3), textcoords='offset points')
ax.set_xscale('log'); ax.set_xlabel('GFLOPs (224×224, log)'); ax.set_ylabel('CV macro-F1 (ortalama)'); ax.set_title('Doğruluk – hesaplama maliyeti', loc='left')
fig.tight_layout(); fig.savefig(os.path.join(RES, 'fig_cv_vs_gflops.png'), dpi=300)

cm = confusion_matrix(y, ENS[SELECTED], labels=range(len(CLASSES)))
fig, ax = plt.subplots(figsize=(5, 4.4)); ax.imshow(cm, cmap='Blues')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=40, ha='right', fontsize=7); ax.set_yticks(range(len(CLASSES)), CLASSES, fontsize=7)
ax.set_xlabel('Tahmin'); ax.set_ylabel('Gerçek'); ax.set_title(f'{SELECTED} — test (5-seed ensemble)', loc='left')
fig.tight_layout(); fig.savefig(os.path.join(RES, 'fig_test_confusion_selected.png'), dpi=300)
print('figürler:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(RES, 'fig_*.png'))))